# Task 5 — A Real Time Series

**Dataset:** `flights.csv` — 144 rows, monthly airline passenger totals 1949–1960  
Columns: `year`, `month`, `passengers`  
Clean data — no missing values.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

df = pd.read_csv('data/flights.csv')
print(f'Shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(df.head(14))

## Step 1 — Build the Time Axis and Plot the Full Series

In [ ]:
# Build an ordered datetime axis
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
month_num = {m: i+1 for i, m in enumerate(month_order)}

df['month_num'] = df['month'].map(month_num)
df['date'] = pd.to_datetime(df[['year', 'month_num']].rename(columns={'month_num': 'month'}).assign(day=1))
df = df.sort_values('date').reset_index(drop=True)

print(df[['date','year','month','passengers']].head())
print(f'\nDate range: {df["date"].min().date()} → {df["date"].max().date()}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(df['date'], df['passengers'], color='steelblue', linewidth=1.8)
ax.fill_between(df['date'], df['passengers'], alpha=0.1, color='steelblue')

ax.set_title('Air Passengers Grew Threefold from 1949 to 1960, with a Strong Repeating Seasonal Cycle',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Monthly Passengers (thousands)')
ax.set_ylim(0)

# Shade alternate years for readability
for yr in range(1949, 1961, 2):
    ax.axvspan(pd.Timestamp(f'{yr}-01-01'), pd.Timestamp(f'{yr+1}-01-01'),
               alpha=0.05, color='gray')

plt.tight_layout()
plt.savefig('task5_full_series.png', dpi=150, bbox_inches='tight')
plt.show()

**The real work:** The month column is a string (`'January'`, `'February'`, …). I converted it to a month number, then combined year and month number into a proper datetime column using `pd.to_datetime`. Without this step the months would sort alphabetically (April, August, December …) and the time axis would be nonsense.

**What the chart shows:** Two patterns at once — a clear long-term upward trend (roughly tripling over 12 years) and a strong repeating within-year cycle. Every summer peaks, every winter troughs.

## Step 2 — Isolate the Seasonal Cycle (One Line per Year)

In [ ]:
years = sorted(df['year'].unique())

# Colour ramp: earlier years are lighter, later years are darker
cmap = plt.cm.Blues
year_colors = {yr: cmap(0.3 + 0.7 * (yr - min(years)) / (max(years) - min(years)))
               for yr in years}

fig, ax = plt.subplots(figsize=(11, 6))

for yr in years:
    yr_data = df[df['year'] == yr].sort_values('month_num')
    ax.plot(yr_data['month_num'], yr_data['passengers'],
            color=year_colors[yr], linewidth=1.6, alpha=0.85)
    # Label the last month of each year on the right
    last = yr_data.iloc[-1]
    ax.text(12.1, last['passengers'], str(yr), color=year_colors[yr],
            fontsize=7, va='center')

ax.set_title('The Same Seasonal Shape Repeats Every Year — Summer Peak, Winter Trough — and Grows Each Year',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Monthly Passengers (thousands)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_xlim(1, 13.5)
ax.set_ylim(0)

# Colour legend for time direction
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=cmap(0.3), label='1949 (lightest)'),
    Patch(color=cmap(1.0), label='1960 (darkest)')
], loc='upper left', fontsize=9, title='Year')

plt.tight_layout()
plt.savefig('task5_seasonal_lines.png', dpi=150, bbox_inches='tight')
plt.show()

**Why this chart?**  
Placing all 12 years on the same month-axis isolates the repeating seasonal pattern by removing the long-term trend from the horizontal axis. Each line traces one year's within-year journey. The blue colour gradient encodes time progression — lighter for earlier years, darker for later — so the reader can immediately see that every year's line is higher than the previous one (the growth trend) while the shape stays consistent.

## Step 3 — Which Month is Consistently the Busiest?

In [ ]:
monthly_avg = df.groupby('month_num').agg(
    mean_passengers=('passengers', 'mean'),
    month_name=('month', 'first')
).reset_index().sort_values('month_num')

# Colour July (peak month) differently
bar_colors = ['#E15759' if m == 7 else '#4E79A7' for m in monthly_avg['month_num']]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(monthly_avg['month_name'], monthly_avg['mean_passengers'],
              color=bar_colors, edgecolor='white')

# Label bars
for bar, val in zip(bars, monthly_avg['mean_passengers']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'{val:.0f}', ha='center', va='bottom', fontsize=8)

ax.set_title('July is Consistently the Busiest Month — Averaging 56% More Passengers than January',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Average Monthly Passengers (thousands)')
ax.set_ylim(0)
ax.tick_params(axis='x', rotation=30)

# Annotate July
july_val = monthly_avg[monthly_avg['month_num'] == 7]['mean_passengers'].values[0]
ax.annotate('July\n(peak)', xy=(6, july_val), xytext=(7, july_val + 20),
            arrowprops=dict(arrowstyle='->', color='#E15759'), fontsize=9, color='#E15759', ha='center')

plt.tight_layout()
plt.savefig('task5_busiest_month.png', dpi=150, bbox_inches='tight')
plt.show()

**Why a bar chart?**  
The question asks for a comparison of a quantity (average passengers) across 12 discrete categories (months). A bar chart is ideal — it makes ranking immediately obvious (compare bar heights), starts at zero (the correct baseline for a count), and uses colour purposefully (red for July, the answer to the question). A line chart would imply a continuous variable; months are categories.

## Step 4 — Which Chart Belongs in a Report?

**Chart 3 (bar chart of busiest month) belongs in a report.** It answers a single, specific question with one clear takeaway: July is the busiest month. A report reader can extract the message in five seconds.

**Chart 1 (full time series)** is useful for exploration and for showing the analyst that both a trend and a cycle exist, but it does not isolate either for the reader. It is a diagnostic chart, not a communication chart. It belongs in a supplementary appendix at most.

**Chart 2 (one line per year)** is a sophisticated analyst's chart — it rewards careful study but asks the reader to hold 12 lines in working memory. Without specific annotation or a guided narrative, it is too dense for a standalone report exhibit.

---

## Written Answer — Is the Seasonal Swing Growing or Just the Airline?

Looking at Chart 2, the absolute gap between the July peak and January trough gets larger every year. In 1949, July exceeds January by about 40,000 passengers; by 1960 the gap has grown to roughly 200,000 passengers. This *looks* like seasonality is intensifying.

**However**, the airline is also growing rapidly — total passengers roughly tripled over 12 years. A seasonal swing that is a constant *proportion* of the total will still get larger in absolute terms as the total grows.

**Chart that would settle it:** Compute the seasonal index for each month and year:  
`seasonal_index = monthly_passengers / annual_average_passengers`  
Then plot a line per year on the same month axis as Chart 2, but with the *index* (not raw passengers) on the y-axis. If the lines all overlap — same shape at roughly the same height — the swing is a stable proportion and growth explains everything. If later years show systematically higher peaks and lower troughs in index terms, then true seasonality is growing.

This is a **normalised seasonal profile chart** (also called a seasonal decomposition plot). It separates the trend component from the seasonal component, which is exactly what the question requires.